1) โหลด speech VAD (จาก CSV ที่เราเพิ่งสร้าง)

In [1]:
import pandas as pd
import numpy as np

speech_csv = r"c:\Users\Legion 5 Pro\OneDrive\Documents\Graduate research\test\wagner\emotion_results_wavlm_all.csv"
df_speech = pd.read_csv(speech_csv)

df_speech


,filename,dialogue_id,utterance_id,arousal,valence,dominance
0,dialogue_1_utterance_1.wav,1,1,0.999705,0.055926,0.957152
1,dialogue_1_utterance_2.wav,1,2,1.006840,0.031109,0.989564
2,dialogue_1_utterance_3.wav,1,3,0.985706,0.109021,0.956718
3,dialogue_1_utterance_4.wav,1,4,0.844993,0.054272,0.860199
4,dialogue_1_utterance_5.wav,1,5,0.531643,0.348859,0.586494
5,dialogue_1_utterance_6.wav,1,6,0.361074,0.336691,0.438390


2) โหลด text VAD
จากตัวอย่างที่คุณแปะมา df_text มีคอลัมน์:

utterance_id

text

valence_text, arousal_text, dominance_text

สมมติอยู่ในไฟล์:

In [2]:
text_csv = r"c:\Users\Legion 5 Pro\OneDrive\Documents\Graduate research\test\park\dialogue_1_vad_text.csv"
df_text = pd.read_csv(text_csv)

df_text


,utterance_id,text,valence_text,arousal_text,dominance_text
0,1,I’m so tired! This player just keeps running a...,2.423500,3.971787,2.804366
1,2,Honestly? A solid 9. It’s boring and feels lik...,2.541238,3.633606,3.026369
2,3,"Both! A good player should stand their ground,...",2.839795,3.533370,3.609603
3,4,You mean you're saying they're running because...,2.743749,3.480893,3.065911
4,5,"I see. When you put it like that, chasing them...",2.702262,3.347695,3.295031
5,6,"Yeah thanks, I will try this new tactic, maybe...",3.264559,3.282050,3.261967


3) สเกล speech logits ไปช่วง [-1,1]

In [3]:
cols = ["arousal", "dominance", "valence"]

for c in cols:
    x = df_speech[c].values
    x_min, x_max = x.min(), x.max()
    if np.isclose(x_max, x_min):
        df_speech[c + "_scaled"] = 0.0
    else:
        x_01 = (x - x_min) / (x_max - x_min)
        df_speech[c + "_scaled"] = x_01 * 2 - 1

df_speech[["arousal_scaled", "dominance_scaled", "valence_scaled"]].describe()


,arousal_scaled,dominance_scaled,valence_scaled
count,6.000000,6.000000,6.000000
mean,0.323244,0.305200,-0.214032
std,0.857208,0.835592,0.925113
min,-1.000000,-1.000000,-1.000000
25%,-0.229113,-0.214295,-0.851601
50%,0.716646,0.705698,-0.676699
75%,0.967064,0.881996,0.565156
max,1.000000,1.000000,1.000000


4) สเกล text VAD ไปช่วง [-1,1]

In [4]:
cols_text = ["valence_text", "arousal_text", "dominance_text"]

for c in cols_text:
    x = df_text[c].values
    x_min, x_max = x.min(), x.max()
    if np.isclose(x_max, x_min):
        df_text[c.replace("_text", "_scaled")] = 0.0
    else:
        x_01 = (x - x_min) / (x_max - x_min)
        df_text[c.replace("_text", "_scaled")] = x_01 * 2 - 1

df_text[["valence_scaled", "arousal_scaled", "dominance_scaled"]].describe()


,valence_scaled,arousal_scaled,dominance_scaled
count,6.000000,6.000000,6.000000
mean,-0.217611,-0.247491,-0.073957
std,0.693530,0.712920,0.686683
min,-1.000000,-1.000000,-1.000000
25%,-0.624297,-0.713096,-0.424048
50%,-0.287788,-0.347342,-0.106914
75%,-0.067167,-0.053271,0.198154
max,1.000000,1.000000,1.000000


5) ตั้งชื่อให้ไม่ชนกัน แล้ว merge

In [5]:
import pandas as pd
import numpy as np

# 1) เตรียมให้ utterance_id ตรงกัน (dialogue 1)
speech = df_speech[df_speech["dialogue_id"] == 1].sort_values("utterance_id").reset_index(drop=True)
text   = df_text.sort_values("utterance_id").reset_index(drop=True)

assert (speech["utterance_id"].to_numpy() == text["utterance_id"].to_numpy()).all()

# 2) ดึงเฉพาะ valence + arousal ที่สเกลแล้ว [-1,1]
df_dissonance = pd.DataFrame({
    "utterance_id": speech["utterance_id"],
    "aro_s": speech["arousal_scaled"].to_numpy(),
    "val_s": speech["valence_scaled"].to_numpy(),
    "aro_t": text["arousal_scaled"].to_numpy(),
    "val_t": text["valence_scaled"].to_numpy(),
})


6) คำนวณ dissonance (เวอร์ชันที่กันบั๊ก dtype แล้ว)

In [6]:
# 3) delta ต่อมิติ
df_dissonance["delta_arousal"] = (df_dissonance["aro_s"] - df_dissonance["aro_t"]).abs()
df_dissonance["delta_valence"] = (df_dissonance["val_s"] - df_dissonance["val_t"]).abs()

# 4) threshold บน [-1,1]
ARO_THR = 0.5
VAL_THR = 0.5

df_dissonance["dissonant_arousal"] = df_dissonance["delta_arousal"] > ARO_THR
df_dissonance["dissonant_valence"] = df_dissonance["delta_valence"] > VAL_THR
df_dissonance["dissonant_any"]     = (
    df_dissonance["dissonant_arousal"] | df_dissonance["dissonant_valence"]
)

# 5) optional: score เดียว (L2 จาก 2 มิติ)
df_dissonance["dissonance_l2"] = np.sqrt(
    df_dissonance["delta_arousal"]**2 + df_dissonance["delta_valence"]**2
)

In [7]:
df_dissonance

,utterance_id,aro_s,val_s,aro_t,val_t,delta_arousal,delta_valence,dissonant_arousal,dissonant_valence,dissonant_any,dissonance_l2
0,1,0.977904,-0.843796,1.000000,-1.000000,0.022096,0.156204,False,False,False,0.157759
1,2,1.000000,-1.000000,0.019391,-0.720024,0.980609,0.279976,True,False,True,1.019794
2,3,0.934547,-0.509602,-0.271260,-0.010069,1.205806,0.499534,True,False,True,1.305183
3,4,0.498745,-0.854202,-0.423423,-0.238461,0.922168,0.615741,True,True,True,1.108842
4,5,-0.471732,1.000000,-0.809653,-0.337115,0.337922,1.337115,False,True,True,1.379155
5,6,-1.000000,0.923409,-1.000000,1.000000,0.000000,0.076591,False,False,False,0.076591
